# *<center>V04 · Time of flight</center>*

**Purpose.** Certify the pipeline's **timing fidelity** — the quantity
every future resolution claim rests on. Four contracts: (1) flight time
in a uniform field against the closed form, with the dt-convergence
table; (2) the √m calibration law; (3) the Wiley–McLaren two-stage
space focus against the piecewise analytic of the **as-built**
geometry; (4) mirror energy focusing dT/dK on the two shipped
reflectron examples (hard vs time-focused).

```
PROVENANCE
  origin   : validation series
  template : V01/V03/V05/V06 (declared bands from measurement)
```

### Conventions
* **Units are mm, V, µs, eV**; CAPITALS are parameters you may change.
* Every threshold is declared before its measurement and asserted.

---

### Assumptions (explicit)
1. **Timing is read by trace interpolation, never `status['tof']`.**
   (AMENDED: the splat fix — nearest-node
   shell + bisection backtrack — made status tof
   dt-STABLE and sub-ns precise; before it, termination was
   label-interpolation-quantized at ~5×10⁻³ and non-convergent.) The
   remaining status-tof bias is purely geometric: it times arrival at
   the h/2 nearest-node SHELL, not the spec face (~3×10⁻³ here at
   h = 0.1 mm over 8.5 mm). Precision timing = quadratic fit of the
   recorded coordinate through the detector plane (exact for
   locally-uniform acceleration), solver-tolerance-limited
   (~3×10⁻⁷ at tol 1e-4 V on 1 kV).
2. **Analytic models describe the as-built node geometry.** Grid
   electrodes are 0.1 mm slabs (one voxel row), field-free inside,
   with region boundaries at the slab faces — not zero-thickness
   planes. Modeling the ideal instead of the built artifact costs
   5×10⁻³ (measured); this is the display==computation doctrine
   applied to theory.
3. **Vacuum, single cold ions** (thermal turnaround spread is a
   source-physics topic, out of scope here; the WM focus is measured
   over a *position* band at rest).
4. **`is_grid` is an ideal transparent equipotential plane** — no field
   penetration, no scattering, 100% transmission. Real-mesh effects are
   a declared model boundary, not validated here.


## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the pusher and two-grid source, the field-free tube, the ring-ladder mirror, and the detector beside the source, with example ions tracing the full chevron.

Deck: `examples/reflectron_tof_oa_refined_wm_matched.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/reflectron_tof_oa_refined_wm_matched.json', banked='panel_tof.png', height=760)


In [ ]:
import json
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

from ion_gym.io.sim_spec import (SimSpec, GeometrySpec, ElectrodeSpec,
                                 ShapeSpec, SourceSpec, IntegrationSpec,
                                 CollisionSpec)
from ion_gym.physics.symmetry import SymmetrySpec
from ion_gym.physics.sim_build import build_run

AMU = 1.66053906660e-27
E_CHG = 1.602176634e-19

# ---- parameters (CAPITALS are yours to change) ----
MZ = 100.0
PITCH = 0.1
DT_NS = 1.0
# uniform cell (the V01-certified plate pattern)
CELL_W = 10.0
CELL_H = 10.0
PLATE_T = 0.5
V_ACCEL = -1000.0        # top-plate volts (negative pulls a +ion upward)
Y_BIRTH = 1.0
Y_DET = 8.0
# Wiley-McLaren stack
WM_H = 30.0
Y_REP, Y_G1, Y_G2 = 0.5, 5.5, 10.5
T_GRID = 0.1             # grid slab thickness = one voxel row
V_REP = 2000.0
WM_Y0S = np.array([2.5, 2.75, 3.0, 3.25, 3.5])   # source position band
L_TARGET = 15.0          # aim the analytic focus ~here (design scan)

m_kg = MZ * AMU

def fly_one(sp):
    errs = sp.validate()
    assert not errs, errs
    model, f, cols, births = build_run(sp)
    tr, status = f(0)
    return np.asarray(tr), {cc: j for j, cc in enumerate(cols)}, status

def t_cross(tr, ci, det, col="y", after_apex=False):
    """Interpolated crossing time of coordinate col through det:
    quadratic fit of the recorded samples around the crossing (exact
    for locally-uniform acceleration). after_apex=True takes the
    RETURN crossing (mirror flights)."""
    t = tr[:, ci["t"]]
    u = tr[:, ci[col]]
    if after_apex:
        i0 = int(np.argmax(u))
        t, u = t[i0:], u[i0:]
        k = int(np.argmax(u <= det))
    else:
        k = int(np.argmax(u >= det))
    if k == 0:
        return np.nan
    s = slice(max(0, k - 2), min(len(u), k + 2))
    co = np.polyfit(t[s], u[s] - det, 2)
    r = np.roots(co)
    r = r[np.isreal(r)].real
    return r[np.argmin(np.abs(r - t[k]))]

def plate_cell(v_top, mz, y0, dt_ns, rec_every=10, t_max=20.0):
    geom = GeometrySpec(
        width_mm=CELL_W, height_mm=CELL_H, mm_per_gu=PITCH,
        symmetry=SymmetrySpec(coords="xyz"),
        electrodes=[
            ElectrodeSpec(name="bot", dc=0.0, shapes=[
                ShapeSpec(type="rect", params={"x_mm": 0, "y_mm": 0,
                    "width_mm": CELL_W, "height_mm": PLATE_T})]),
            ElectrodeSpec(name="top", dc=v_top, shapes=[
                ShapeSpec(type="rect", params={"x_mm": 0,
                    "y_mm": CELL_H - PLATE_T, "width_mm": CELL_W,
                    "height_mm": PLATE_T})])])
    return SimSpec(
        name="V04 timing cell", geometry=geom,
        source=SourceSpec(n_ions=1, distribution="point",
                          x0_mm=CELL_W / 2, y0_mm=y0, ke_lo=0.0,
                          ke_hi=0.0, temperature_k=0.0, mz_list=[mz],
                          tob_span_us=0.0),
        collisions=CollisionSpec(enabled=False),
        integration=IntegrationSpec(t_max_us=t_max, dt_ns=dt_ns,
                                    rec_every=rec_every))

print("helpers ready (deterministic: single cold ions, no RNG)")

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


## 1 · Uniform-field timing and the two estimators

An ion from rest in the plate cell's uniform field E = V/gap; the
closed form is t = √(2d/a). Two readouts of the same flights:

* **Gate A:** the interpolated crossing at y = 8 mm satisfies
  |t/t_analytic − 1| < **10⁻⁵** at every dt ≤ 2 ns (measured 2–4×10⁻⁷;
  the band gives ~30× headroom over the solver-tolerance floor).
* The **status tof table** (termination on the top plate) is printed
  alongside: dt-stable since the splat fix, but biased by
  the h/2 nearest-node shell (~3×10⁻³ here). Not asserted —
  *documented*, so nobody builds a resolution claim on it.

In [ ]:
E_vmm = abs(V_ACCEL) / (CELL_H - 2 * PLATE_T)
a_mm_us2 = E_CHG * (E_vmm * 1e3) / m_kg * 1e-9
t_an = np.sqrt(2 * (Y_DET - Y_BIRTH) / a_mm_us2)
print(f"analytic: E = {E_vmm:.4f} V/mm, t({Y_DET} mm) = {t_an:.9f} us")
rows = []
for dt in (4.0, 2.0, 1.0, 0.5):
    tr, ci, st = fly_one(plate_cell(V_ACCEL, MZ, Y_BIRTH, dt))
    tc = t_cross(tr, ci, Y_DET)
    rows.append((dt, tc, (tc - t_an) / t_an, st["tof"]))
    print(f"  dt={dt:4.1f} ns: interp rel err = {(tc-t_an)/t_an:+.2e}   "
          f"status tof = {st['tof']:.6f} us (grid-biased)")
errs = np.array([abs(r[2]) for r in rows if r[0] <= 2.0])
PASS_A = bool(np.all(errs < 1e-5))
print("PASS" if PASS_A else "FAIL",
      "— A: interpolated timing at the solver-tolerance floor; "
      "status tof documented as grid-quantized, not used for timing")
assert PASS_A

## 2 · The √m calibration law

Same cell, m/z ladder at fixed drive. t/√(m/z) must be constant —
in a static field this is a pure kinematics identity, so the assert is
a **wiring guard** (the mass the spec declares is the mass the kernel
integrates), at machine level.

* **Gate B:** max |t/√mz ÷ (t/√mz)₁₀₀ − 1| < **10⁻⁹**.

In [ ]:
ts = {}
for mz in (50.0, 100.0, 200.0, 400.0):
    tr, ci, st = fly_one(plate_cell(V_ACCEL, mz, Y_BIRTH, DT_NS))
    ts[mz] = t_cross(tr, ci, Y_DET)
r0 = ts[100.0] / np.sqrt(100.0)
dev = max(abs(t / np.sqrt(mz) / r0 - 1) for mz, t in ts.items())
for mz, t in sorted(ts.items()):
    print(f"  mz={mz:5.0f}: t = {t:.6f} us, t/sqrt(mz) = "
          f"{t/np.sqrt(mz):.9f}")
PASS_B = dev < 1e-9
print("PASS" if PASS_B else "FAIL",
      f"— B: sqrt(m) law holds to {dev:.1e} (machine level)")
assert PASS_B

## 3 · Wiley–McLaren two-stage space focus

Repeller / grid-1 / grid-2 / drift, ions **from rest across a ±0.5 mm
position band**. The design point (grid-1 voltage) is chosen *by the
notebook itself*: scan the slab-aware piecewise analytic for the V_G1
whose first-order space focus lands nearest L_TARGET — no magic
voltage. The same analytic then predicts absolute times and the focus
plane, and five flights measure both.

* **Gate C:** |t_meas/t_analytic − 1| < **3×10⁻⁴** for every source
  position (measured ≤ 7×10⁻⁵ with the slab-aware model — the
  zero-thickness-grid model misses by 5×10⁻³, assumption 2);
  |L_focus,meas − L_focus,analytic| < **0.5 mm**; and the defocused
  spread at L = 2 mm exceeds the focal spread by > **20×**.

In [ ]:
def an_T(y0, L, vg):
    """Slab-aware piecewise analytic (assumption 2): s-region
    [y0, Y_G1], field-free slab [Y_G1, Y_G1+T_GRID], d-region
    [Y_G1+T_GRID, Y_G2], field-free slab, then drift L."""
    Es = (V_REP - vg) / (Y_G1 - Y_REP)
    Ed = vg / (Y_G2 - (Y_G1 + T_GRID))
    a_s = E_CHG * Es * 1e3 / m_kg * 1e-9
    a_d = E_CHG * Ed * 1e3 / m_kg * 1e-9
    s = Y_G1 - y0
    t1 = np.sqrt(2 * s / a_s)
    v1 = a_s * t1
    dd = Y_G2 - (Y_G1 + T_GRID)
    v2 = np.sqrt(v1**2 + 2 * a_d * dd)
    t2 = (v2 - v1) / a_d
    return t1 + T_GRID / v1 + t2 + T_GRID / v2 + L / v2

# design scan: pick VG landing the analytic focus nearest L_TARGET
Ls = np.linspace(1, 18, 341)
best = None
for vg in np.linspace(200, 1900, 341):
    sp_ = [float(np.ptp(an_T(WM_Y0S, L, vg))) for L in Ls]
    Lst = Ls[int(np.argmin(sp_))]
    if best is None or abs(Lst - L_TARGET) < abs(best[1] - L_TARGET):
        best = (float(vg), float(Lst), min(sp_))
VG, L_AN, sp_an = best
print(f"design point: V_G1 = {VG:.1f} V -> analytic focus "
      f"L* = {L_AN:.2f} mm (spread {sp_an*1e3:.3f} ns)")

def wm_spec(y0):
    geom = GeometrySpec(
        width_mm=CELL_W, height_mm=WM_H, mm_per_gu=PITCH,
        symmetry=SymmetrySpec(coords="xyz"),
        electrodes=[
            ElectrodeSpec(name="rep", dc=V_REP, shapes=[
                ShapeSpec(type="rect", params={"x_mm": 0, "y_mm": 0,
                    "width_mm": CELL_W, "height_mm": Y_REP})]),
            ElectrodeSpec(name="g1", dc=VG, is_grid=True, shapes=[
                ShapeSpec(type="rect", params={"x_mm": 0, "y_mm": Y_G1,
                    "width_mm": CELL_W, "height_mm": T_GRID})]),
            ElectrodeSpec(name="g2", dc=0.0, is_grid=True, shapes=[
                ShapeSpec(type="rect", params={"x_mm": 0, "y_mm": Y_G2,
                    "width_mm": CELL_W, "height_mm": T_GRID})]),
            ElectrodeSpec(name="top", dc=0.0, shapes=[
                ShapeSpec(type="rect", params={"x_mm": 0,
                    "y_mm": WM_H - 0.5, "width_mm": CELL_W,
                    "height_mm": 0.5})])])
    return SimSpec(
        name="V04 WM", geometry=geom,
        source=SourceSpec(n_ions=1, distribution="point",
                          x0_mm=CELL_W / 2, y0_mm=float(y0), ke_lo=0.0,
                          ke_hi=0.0, temperature_k=0.0, mz_list=[MZ],
                          tob_span_us=0.0),
        collisions=CollisionSpec(enabled=False),
        integration=IntegrationSpec(t_max_us=8.0, dt_ns=DT_NS,
                                    rec_every=5))

t0 = time.time()
trs = [fly_one(wm_spec(y0)) for y0 in WM_Y0S]
drift0 = Y_G2 + T_GRID
Lscan = np.linspace(2, 18, 81)
spread = np.array([float(np.ptp(np.array(
    [t_cross(tr, ci, drift0 + L) for tr, ci, st in trs])))
    for L in Lscan])
L_meas = float(Lscan[int(np.argmin(spread))])
tc_ms = np.array([t_cross(tr, ci, drift0 + L_meas) for tr, ci, st in trs])
tc_an = an_T(WM_Y0S, L_meas, VG)
rel = np.abs(tc_ms / tc_an - 1)
print(f"[{time.time()-t0:.0f}s] measured focus L = {L_meas:.2f} mm "
      f"(analytic {L_AN:.2f}); min spread {spread.min()*1e3:.3f} ns; "
      f"defocus/focus = {spread[0]/spread.min():.0f}x")
print(f"absolute-time agreement: max |rel| = {rel.max():.1e}")
PASS_C = (rel.max() < 3e-4 and abs(L_meas - L_AN) < 0.5
          and spread[0] / spread.min() > 20)
print("PASS" if PASS_C else "FAIL",
      "— C: the as-built piecewise analytic predicts both the absolute "
      "times and the space-focus plane")
assert PASS_C

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(Lscan, spread * 1e3, lw=1.2)
ax.axvline(L_AN, color="g", ls="--", lw=1, label="analytic focus")
ax.axvline(L_meas, color="k", ls=":", lw=1, label="measured focus")
ax.set_xlabel("detector plane, mm past G2")
ax.set_ylabel("arrival spread (ns)")
ax.set_title("Wiley-McLaren space focus", fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


## 4 · Mirror energy focusing: dT/dK

The two shipped reflectron examples flown over their own energy window
(760–840 eV), return time read at the source plane by interpolation.
The hard mirror's dT/dK is the uncorrected energy dependence; the
time-focused mirror must null it to first order **inside the window**.

* **Gate D:** |dT/dK|_focused < **0.05·**|dT/dK|_hard (measured
  0.010); the focused T(K) has an interior minimum (first-order focus
  in-window); and spread_hard / spread_focused > **20×** (measured
  ~89×).

In [ ]:
from ion_gym.io.deck_params import describe_deck
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
ks = np.linspace(760.0, 840.0, 5)
res = {}
for name in ("reflectron_tof_r-z_hard_mirror",
             "reflectron_tof_r-z_time-focused_mirror"):
    d = json.load(open(str(ROOT / 'examples' / f'{name}.json')))
    ts = []
    for K in ks:
        sp = SimSpec.from_dict(d)
        # Say what this deck supplied -- drives, ion, gas, integration --
        # so the operating point is visible where the spec is built.
        describe_deck(sp)
        sp.source.distribution = "point"
        sp.source.n_ions = 1
        sp.source.ke_lo = sp.source.ke_hi = float(K)
        sp.source.tob_span_us = 0.0
        sp.integration = IntegrationSpec(t_max_us=30.0, dt_ns=1.0,
                                         rec_every=10)
        tr, ci, st = fly_one(sp)
        ts.append(t_cross(tr, ci, sp.source.x0_mm, col="x",
                          after_apex=True))
    ts = np.array(ts)
    slope = np.polyfit(ks, ts, 1)[0]
    res[name] = (ts, slope)
    print(f"  {name.split('_')[-2]}-{name.split('_')[-1]}: "
          f"dT/dK = {slope*1e3:+.4f} ns/eV, "
          f"spread = {np.ptp(ts)/np.median(ts):.2e}")
t_h, s_h = res["reflectron_tof_r-z_hard_mirror"]
t_f, s_f = res["reflectron_tof_r-z_time-focused_mirror"]
interior_min = bool(0 < int(np.argmin(t_f)) < len(t_f) - 1)
PASS_D = (abs(s_f) < 0.05 * abs(s_h) and interior_min
          and np.ptp(t_h) / np.ptp(t_f) > 20)
print("PASS" if PASS_D else "FAIL",
      "— D: the time-focused mirror nulls dT/dK to first order "
      "in-window; the hard mirror does not")
assert PASS_D

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
axes[0].plot(ks, t_h, "o-")
axes[0].set_title("hard mirror", fontsize=9)
axes[1].plot(ks, t_f, "o-")
axes[1].set_title("time-focused mirror (note the interior minimum)",
                  fontsize=9)
for ax in axes:
    ax.set_xlabel("ion energy (eV)")
    ax.set_ylabel("return time (us)")
plt.tight_layout()
plt.show()


## 5 · Gate T3 — timestep convergence

A fixed-step integrator's defense is not equivalence to anyone else's
stepper — it is a **measured plateau**: fly the same ion at Δt, Δt/2,
Δt/4, Δt/8 and show the observable an experimentalist cares about (the
TOF, here through the time-focused mirror) stops moving. Two asserted
claims: successive |ΔTOF| **shrink monotonically** (the integrator is in
its convergent regime), and the finest halving moves the TOF by **less
than 1 ppm** — a declared numerical error bar the certified numbers can
carry.

In [ ]:
# ---- Gate T3: dt-halving TOF convergence --------------------------------
DT_LADDER_NS = [2.0, 1.0, 0.5, 0.25]     # each entry halves the previous
T3_KE_EV     = 800.0                     # mid-band launch (section 4's ladder)
T3_PPM_MAX   = 1.0                       # plateau bound on the finest halving

d = json.load(open(str(ROOT / 'examples' /
                       'reflectron_tof_r-z_time-focused_mirror.json')))
tofs = []
for dt in DT_LADDER_NS:
    sp = SimSpec.from_dict(d)
    sp.source.distribution = "point"
    sp.source.n_ions = 1
    sp.source.ke_lo = sp.source.ke_hi = T3_KE_EV
    sp.source.tob_span_us = 0.0
    sp.integration = IntegrationSpec(t_max_us=30.0, dt_ns=float(dt),
                                     rec_every=10)
    tr, ci, st = fly_one(sp)
    tofs.append(t_cross(tr, ci, sp.source.x0_mm, col="x", after_apex=True))
tofs = np.array(tofs)
deltas = np.abs(np.diff(tofs))
ppm = deltas / tofs[1:] * 1e6
print(f"{'dt (ns)':>8s} {'TOF (us)':>14s} {'|dTOF| vs prev (ppm)':>22s}")
for k, dt in enumerate(DT_LADDER_NS):
    extra = f"{ppm[k-1]:22.4f}" if k else f"{'—':>22s}"
    print(f"{dt:8.2f} {tofs[k]:14.7f} {extra}")
mono = bool(np.all(np.diff(deltas) < 0))
plateau = bool(ppm[-1] < T3_PPM_MAX)
print(f"\nmonotone shrink: {mono} | finest halving: {ppm[-1]:.4f} ppm "
      f"(bound {T3_PPM_MAX:g} ppm)")
print("PASS" if (mono and plateau) else "FAIL",
      "— T3: TOF plateaus under dt-halving; the declared numerical "
      "error bar is the finest-halving delta above")
assert mono and plateau, (
    f"Gate T3 failed: monotone={mono}, finest halving {ppm[-1]:.4f} ppm "
    f"(needs < {T3_PPM_MAX:g}). MEASURED values — if the physics or the "
    f"deck changed, re-derive the ladder before touching the bound.")


---
## Summary

| Gate | Claim | Band | Result |
|---|---|---|---|
| A | uniform-field timing (interpolated) | <1e-5 | 2–4×10⁻⁷ |
| B | √m law (wiring guard) | <1e-9 | machine |
| C | WM focus vs as-built analytic | 3e-4 / 0.5 mm / 20× | 7e-5 / 0.05 mm / 43× |
| D | mirror dT/dK | 0.05× / interior min / 20× | 0.010× / ✓ / 89× |

**What this buys the instrument work:** timing claims must come from
trace interpolation (status tof times the h/2 nearest-node shell, a
~3×10⁻³ geometric bias at this pitch — a documented readout property,
not an integrator defect); analytic
design models must describe the as-built node geometry (5×10⁻³
penalty for idealizing the grids away); and the shipped time-focused
mirror example is *certified* first-order energy-focusing at its tune
(operating point: 760–840 eV window, m/z 100, its own JSON voltages).

### Citations
[1] W. C. Wiley, I. H. McLaren, *Rev. Sci. Instrum.* **26**, 1150
(1955) — two-stage space focusing.
[2] B. A. Mamyrin *et al.*, *Sov. Phys. JETP* **37**, 45 (1973) — the
reflectron and energy focusing.
[3] M. Guilhaus, *J. Mass Spectrom.* **30**, 1519 (1995) — TOF
principles tutorial (timing error budgets).
[4] Yavor, *Optics of Charged Particle Analyzers* — mirror aberration
theory.
[5] Verenchikov, *Mass Spectrom. Rev.* (2024) — multi-reflecting TOF
perspective (project reference set).


## Read-out

Timing fidelity underwrites every resolution claim the toolkit will ever make. The √m law is a pure kinematic identity in a static field, so any departure is numerical, not physical — which makes it a sensitive probe of step-size bias. Quote the step size with every timing number.